## Cell 1 - Setup and GPU Configuration

Mount Google Drive, install required packages, set random seeds for reproducibility, detect the available device (GPU/CPU), and define all file paths used throughout the notebook.

In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install packages
import subprocess
subprocess.run(['pip', 'install', '-q', 'torch', 'torchvision', 'scikit-learn', 'matplotlib', 'numpy'])

import torch
import numpy as np

# Set seeds
torch.manual_seed(42)
np.random.seed(42)

# GPU info
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch version: {torch.__version__}')

# Paths
DRIVE_ROOT = '/content/drive/MyDrive/ATML_FOLDER'
NPZ_PATH = f'{DRIVE_ROOT}/slice_dataset.npz'
META_PATH = f'{DRIVE_ROOT}/slice_metadata.csv'
SAVE_DIR = DRIVE_ROOT
print(f'Drive root: {DRIVE_ROOT}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB
PyTorch version: 2.10.0+cu128
Drive root: /content/drive/MyDrive/ATML_FOLDER


## Cell 2 - Data Loading and Preparation

Load the pre-built `slice_dataset.npz` (296 subjects, each with 3 tissue types × 3 planes × 64×64 pixels) and the accompanying metadata CSV. Flatten the data into 888 individual samples (subject × tissue), wrap them in a PyTorch `Dataset`, and split into 800 training / 88 validation samples with reproducible `DataLoader`s.

In [3]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, random_split

# Load NPZ - 296 subjects, each (3, 3, 64, 64): tissue x plane x H x W
print('Loading slice_dataset.npz...')
npz = np.load(NPZ_PATH, allow_pickle=True)
subject_ids = list(npz.keys())
print(f'Subjects loaded: {len(subject_ids)}')
print(f'Shape per subject: {npz[subject_ids[0]].shape}')

# Load metadata
meta_df = pd.read_csv(META_PATH)
print(f'Metadata: {len(meta_df)} rows, columns: {list(meta_df.columns)}')

# Build flat sample list: each sample = (subject_id, tissue_idx, array(3,64,64))
# tissue_idx: 0=CSF, 1=GM, 2=WM
TISSUE_NAMES = ['CSF', 'GM', 'WM']
samples = []
for sid in subject_ids:
    arr = npz[sid]  # (3, 3, 64, 64)
    for t in range(3):
        samples.append((sid, t, arr[t].astype(np.float32)))  # (3, 64, 64)

print(f'Total samples (296 x 3): {len(samples)}')
print(f'Sample shape: {samples[0][2].shape}')

class BrainSliceDataset(Dataset):
    def __init__(self, sample_list):
        self.samples = sample_list
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        sid, tissue_idx, arr = self.samples[idx]
        return torch.tensor(arr, dtype=torch.float32), sid, tissue_idx

full_dataset = BrainSliceDataset(samples)

# Split 800 train / 88 val
torch.manual_seed(42)
train_ds, val_ds = random_split(full_dataset, [800, 88])
print(f'Train samples: {len(train_ds)}')
print(f'Val samples:   {len(val_ds)}')

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# Confirm one batch
batch_x, batch_sid, batch_t = next(iter(train_loader))
print(f'One batch shape: {batch_x.shape}')
print(f'Value range: [{batch_x.min():.3f}, {batch_x.max():.3f}]')

Loading slice_dataset.npz...
Subjects loaded: 296
Shape per subject: (3, 3, 64, 64)
Metadata: 296 rows, columns: ['subject_id', 'disc', 'CDR', 'AD_label']
Total samples (296 x 3): 888
Sample shape: (3, 64, 64)
Train samples: 800
Val samples:   88
One batch shape: torch.Size([32, 3, 64, 64])
Value range: [-2.366, 2.883]


## Cell 3 - VAE Model Architecture

Define the `BrainVAE` convolutional Variational Autoencoder. The encoder compresses 3×64×64 brain slice images (axial/coronal/sagittal channels) down to a 64-dimensional latent space via three strided conv layers and a fully-connected bottleneck. The decoder mirrors this with transposed convolutions back to 3×64×64. A reparameterisation trick enables end-to-end gradient flow. A dummy forward pass verifies all tensor shapes.

In [4]:
import torch
import torch.nn as nn

class BrainVAE(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        self.latent_dim = latent_dim

        # Encoder: 3x64x64 -> 128x8x8 -> 256 -> mu/logvar
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1),   # 32x32x32
            nn.BatchNorm2d(32), nn.LeakyReLU(0.2),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),  # 64x16x16
            nn.BatchNorm2d(64), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), # 128x8x8
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
        )
        self.encoder_fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256),
            nn.LeakyReLU(0.2),
        )
        self.fc_mu     = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)

        # Decoder: latent_dim -> 256 -> 128x8x8 -> 3x64x64
        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 128 * 8 * 8),
            nn.LeakyReLU(0.2),
        )
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 3, stride=2, padding=1, output_padding=1),  # 64x16x16
            nn.BatchNorm2d(64), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),   # 32x32x32
            nn.BatchNorm2d(32), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(32, 3, 3, stride=2, padding=1, output_padding=1),    # 3x64x64
            nn.Tanh(),
        )

    def encode(self, x):
        h = self.encoder_fc(self.encoder_conv(x))
        return self.fc_mu(h)  # returns mu only - used for embedding extraction

    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu  # deterministic at inference

    def decode(self, z):
        h = self.decoder_fc(z).view(-1, 128, 8, 8)
        return self.decoder_conv(h)

    def forward(self, x):
        h = self.encoder_fc(self.encoder_conv(x))
        mu     = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        z      = self.reparameterize(mu, logvar)
        recon  = self.decode(z)
        return recon, mu, logvar

# Verify output shape with dummy forward pass
model = BrainVAE(latent_dim=64).to(device)
dummy = torch.randn(4, 3, 64, 64).to(device)
recon, mu, logvar = model(dummy)
print(f'Input shape:       {dummy.shape}')
print(f'Reconstruction:    {recon.shape}   <- must be (4, 3, 64, 64)')
print(f'mu shape:          {mu.shape}      <- must be (4, 64)')
print(f'logvar shape:      {logvar.shape}  <- must be (4, 64)')
assert recon.shape == dummy.shape, 'Output shape mismatch!'
print('Shape check passed.')

total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

Input shape:       torch.Size([4, 3, 64, 64])
Reconstruction:    torch.Size([4, 3, 64, 64])   <- must be (4, 3, 64, 64)
mu shape:          torch.Size([4, 64])      <- must be (4, 64)
logvar shape:      torch.Size([4, 64])  <- must be (4, 64)
Shape check passed.
Total parameters: 4,439,299


## Cell 4 - Loss Function and Training Configuration

Define the VAE loss as a weighted sum of pixel-wise MSE reconstruction loss and the KL-divergence regularisation term (β = 0.5). Set up the Adam optimiser with weight decay, a `ReduceLROnPlateau` learning-rate scheduler, and early-stopping hyperparameters.

In [6]:
import torch.optim as optim

BETA       = 0.5   # KL weight
LR         = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 100
PATIENCE   = 15

def vae_loss(recon, x, mu, logvar, beta=BETA):
    recon_loss = nn.functional.mse_loss(recon, x, reduction='mean')
    kl_loss    = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    total      = recon_loss + beta * kl_loss
    return total, recon_loss, kl_loss

optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.5
)

print(f'Beta (KL weight):  {BETA}')
print(f'Learning rate:     {LR}')
print(f'Max epochs:        {MAX_EPOCHS}')
print(f'Early stop patience: {PATIENCE}')
print('Loss function and optimizer ready.')

Beta (KL weight):  0.5
Learning rate:     0.001
Max epochs:        100
Early stop patience: 15
Loss function and optimizer ready.


## Cell 5 - Training Loop

Run the full training loop for up to 100 epochs with early stopping (patience = 15). Each epoch computes train and validation losses, saves the best model weights to `best_vae.pth`, logs a reconstruction visualisation every 10 epochs, and plots the final loss curve.

In [7]:
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

train_losses, val_losses = [], []
best_val_loss = float('inf')
patience_counter = 0
best_epoch = 0

def save_reconstructions(model, val_loader, epoch, save_dir):
    model.eval()
    with torch.no_grad():
        batch_x, _, _ = next(iter(val_loader))
        batch_x = batch_x[:4].to(device)
        recon, _, _ = model(batch_x)
        batch_x = batch_x.cpu().numpy()
        recon   = recon.cpu().numpy()

    fig, axes = plt.subplots(4, 6, figsize=(18, 12))
    channel_names = ['Axial', 'Coronal', 'Sagittal']
    for i in range(4):
        for c in range(3):
            axes[i, c].imshow(batch_x[i, c], cmap='gray')
            axes[i, c].set_title(f'Orig {channel_names[c]}', fontsize=8)
            axes[i, c].axis('off')
            axes[i, c+3].imshow(recon[i, c], cmap='gray')
            axes[i, c+3].set_title(f'Recon {channel_names[c]}', fontsize=8)
            axes[i, c+3].axis('off')
    plt.suptitle(f'Epoch {epoch} Reconstructions', fontsize=14, fontweight='bold')
    plt.tight_layout()
    path = os.path.join(save_dir, f'vae_reconstruction_epoch_{epoch}.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'  Saved reconstruction: {path}')

print('Starting training...')
print(f'{"Epoch":>6} | {"Train Loss":>11} | {"Val Loss":>10} | {"Recon":>10} | {"KL":>8} | {"LR":>10}')
print('-' * 70)

for epoch in range(1, MAX_EPOCHS + 1):
    # --- Train ---
    model.train()
    epoch_train_loss = 0.0
    for batch_x, _, _ in train_loader:
        batch_x = batch_x.to(device)
        optimizer.zero_grad()
        recon, mu, logvar = model(batch_x)
        loss, recon_l, kl_l = vae_loss(recon, batch_x, mu, logvar)
        loss.backward()
        optimizer.step()
        epoch_train_loss += loss.item() * batch_x.size(0)
    epoch_train_loss /= len(train_loader.dataset)

    # --- Validate ---
    model.eval()
    epoch_val_loss = 0.0
    epoch_recon_l  = 0.0
    epoch_kl_l     = 0.0
    with torch.no_grad():
        for batch_x, _, _ in val_loader:
            batch_x = batch_x.to(device)
            recon, mu, logvar = model(batch_x)
            loss, recon_l, kl_l = vae_loss(recon, batch_x, mu, logvar)
            epoch_val_loss += loss.item() * batch_x.size(0)
            epoch_recon_l  += recon_l.item() * batch_x.size(0)
            epoch_kl_l     += kl_l.item() * batch_x.size(0)
    epoch_val_loss /= len(val_loader.dataset)
    epoch_recon_l  /= len(val_loader.dataset)
    epoch_kl_l     /= len(val_loader.dataset)

    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)
    scheduler.step(epoch_val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    print(f'{epoch:>6} | {epoch_train_loss:>11.5f} | {epoch_val_loss:>10.5f} | {epoch_recon_l:>10.5f} | {epoch_kl_l:>8.5f} | {current_lr:>10.2e}')

    # Save best model
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_epoch    = epoch
        patience_counter = 0
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, 'best_vae.pth'))
    else:
        patience_counter += 1

    # Reconstruction visualization every 10 epochs
    if epoch % 10 == 0:
        save_reconstructions(model, val_loader, epoch, SAVE_DIR)

    # Early stopping
    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}. Best epoch: {best_epoch}')
        break

# Loss curve
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_losses, label='Train Loss', linewidth=2)
ax.plot(val_losses,   label='Val Loss',   linewidth=2)
ax.axvline(best_epoch - 1, color='red', linestyle='--', label=f'Best epoch ({best_epoch})')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('VAE Loss', fontsize=12)
ax.set_title('VAE Training Loss Curve', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
loss_curve_path = os.path.join(SAVE_DIR, 'vae_loss_curve.png')
plt.savefig(loss_curve_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'\nLoss curve saved: {loss_curve_path}')
print(f'Best val loss: {best_val_loss:.5f} at epoch {best_epoch}')

Starting training...
 Epoch |  Train Loss |   Val Loss |      Recon |       KL |         LR
----------------------------------------------------------------------
     1 |     0.19829 |    0.11222 |    0.11098 |  0.00248 |   1.00e-03
     2 |     0.11802 |    0.10583 |    0.09943 |  0.01281 |   1.00e-03
     3 |     0.10408 |    0.08849 |    0.07816 |  0.02066 |   1.00e-03
     4 |     0.09552 |    0.07353 |    0.06087 |  0.02533 |   1.00e-03
     5 |     0.08886 |    0.07189 |    0.05716 |  0.02946 |   1.00e-03
     6 |     0.08888 |    0.06597 |    0.05269 |  0.02656 |   1.00e-03
     7 |     0.08553 |    0.06467 |    0.04875 |  0.03183 |   1.00e-03
     8 |     0.08403 |    0.06110 |    0.04951 |  0.02317 |   1.00e-03
     9 |     0.08331 |    0.06391 |    0.04979 |  0.02824 |   1.00e-03
    10 |     0.08239 |    0.06306 |    0.04649 |  0.03314 |   1.00e-03
  Saved reconstruction: /content/drive/MyDrive/ATML_FOLDER/vae_reconstruction_epoch_10.png
    11 |     0.08162 |    0.06117 | 

## Cell 6 - Post-Training Validation and Anomaly Detection

Reload the best saved model and compute per-sample reconstruction error (MSE) across all 888 samples. Aggregate to per-subject mean error, join with metadata labels, and compare healthy vs. dementia groups using a Mann-Whitney U test. Plot an anomaly-score histogram showing the separation between groups.

In [8]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import os

# Load best model
model.load_state_dict(torch.load(os.path.join(SAVE_DIR, 'best_vae.pth'), map_location=device))
model.eval()
print('Best model loaded.')

# Compute reconstruction error for all 888 samples
all_errors = []  # list of (subject_id, tissue_idx, mse)
with torch.no_grad():
    for batch_x, batch_sid, batch_t in DataLoader(full_dataset, batch_size=32, shuffle=False):
        batch_x = batch_x.to(device)
        recon, _, _ = model(batch_x)
        mse = ((recon - batch_x) ** 2).mean(dim=[1, 2, 3])  # per-sample MSE
        for i in range(len(batch_sid)):
            all_errors.append({
                'subject_id': batch_sid[i],
                'tissue_idx': batch_t[i].item(),
                'recon_error': mse[i].item()
            })

errors_df = pd.DataFrame(all_errors)
print(f'Computed errors for {len(errors_df)} samples')

# Per-subject mean reconstruction error (across 3 tissues)
subject_errors = errors_df.groupby('subject_id')['recon_error'].mean().reset_index()
subject_errors.columns = ['subject_id', 'mean_recon_error']

# Join with metadata
meta_df = pd.read_csv(META_PATH)
subject_errors = subject_errors.merge(meta_df[['subject_id', 'CDR', 'AD_label']], on='subject_id', how='left')

# Filter to labeled subjects only
labeled = subject_errors.dropna(subset=['CDR'])
healthy  = labeled[labeled['AD_label'] == 0]['mean_recon_error'].values
dementia = labeled[labeled['AD_label'] == 1]['mean_recon_error'].values

print(f'\nLabeled subjects: {len(labeled)}')
print(f'Healthy  (n={len(healthy)}): mean error = {healthy.mean():.5f} +/- {healthy.std():.5f}')
print(f'Dementia (n={len(dementia)}): mean error = {dementia.mean():.5f} +/- {dementia.std():.5f}')

# Mann-Whitney U test
stat, pval = stats.mannwhitneyu(healthy, dementia, alternative='two-sided')
print(f'\nMann-Whitney U test: U={stat:.1f}, p={pval:.4f}')
if pval < 0.05:
    print('Result: SIGNIFICANT difference (p < 0.05)')
else:
    print('Result: No significant difference (p >= 0.05)')

# Anomaly score histogram
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(healthy,  bins=25, alpha=0.6, color='steelblue', label=f'Healthy (n={len(healthy)})',  density=True)
ax.hist(dementia, bins=25, alpha=0.6, color='crimson',   label=f'Dementia (n={len(dementia)})', density=True)
ax.set_xlabel('Mean Reconstruction Error (MSE)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('VAE Anomaly Scores: Healthy vs Dementia\n(Lower error = brain reconstructs more normally)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.text(0.98, 0.95, f'p = {pval:.4f}', transform=ax.transAxes,
        ha='right', va='top', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
ax.grid(True, alpha=0.3)
plt.tight_layout()
anomaly_path = os.path.join(SAVE_DIR, 'vae_anomaly_scores.png')
plt.savefig(anomaly_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Anomaly score plot saved: {anomaly_path}')

Best model loaded.
Computed errors for 888 samples

Labeled subjects: 161
Healthy  (n=93): mean error = 0.04741 +/- 0.00451
Dementia (n=68): mean error = 0.04961 +/- 0.00772

Mann-Whitney U test: U=2807.0, p=0.2250
Result: No significant difference (p >= 0.05)
Anomaly score plot saved: /content/drive/MyDrive/ATML_FOLDER/vae_anomaly_scores.png


## Cell 7 - Embedding Extraction and Export

Reload the best model and extract the encoder's mean (μ) latent vector for every labeled subject, producing a (3, 64) embedding per subject (one 64-dim vector per tissue type). Save all embeddings to `vae_embeddings.npz` and the anomaly scores to `vae_anomaly_scores.csv` on Google Drive. Print a final summary of all saved artefacts.

In [9]:
import numpy as np
import pandas as pd
import os

# Load best model
model.load_state_dict(torch.load(os.path.join(SAVE_DIR, 'best_vae.pth'), map_location=device))
model.eval()

# Load metadata to get labeled subject IDs
meta_df = pd.read_csv(META_PATH)
labeled_ids = set(meta_df.dropna(subset=['CDR'])['subject_id'].tolist())
print(f'Extracting embeddings for {len(labeled_ids)} labeled subjects...')

# Extract embeddings: subject_id -> (3, 64) array
embeddings = {}
with torch.no_grad():
    for sid in labeled_ids:
        arr = npz[sid]  # (3, 3, 64, 64)
        x   = torch.tensor(arr, dtype=torch.float32).to(device)  # (3, 3, 64, 64)
        mu  = model.encode(x)   # (3, 64)
        embeddings[sid] = mu.cpu().numpy()

print(f'Embeddings extracted: {len(embeddings)} subjects')
print(f'Embedding shape per subject: {next(iter(embeddings.values())).shape}')

# Save embeddings NPZ
emb_path = os.path.join(SAVE_DIR, 'vae_embeddings.npz')
np.savez_compressed(emb_path, **embeddings)
emb_size_mb = os.path.getsize(emb_path) / 1e6
print(f'vae_embeddings.npz saved: {emb_size_mb:.2f} MB')

# Save anomaly scores CSV
scores_path = os.path.join(SAVE_DIR, 'vae_anomaly_scores.csv')
subject_errors.to_csv(scores_path, index=False)
print(f'vae_anomaly_scores.csv saved')

# Summary
print('\n' + '='*50)
print('ALL OUTPUTS SAVED TO GOOGLE DRIVE:')
print('='*50)
for fname in ['best_vae.pth', 'vae_embeddings.npz', 'vae_anomaly_scores.csv',
              'vae_loss_curve.png', 'vae_anomaly_scores.png']:
    fpath = os.path.join(SAVE_DIR, fname)
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f'  {fname}: {size_mb:.2f} MB')
    else:
        print(f'  {fname}: NOT FOUND')

Extracting embeddings for 161 labeled subjects...
Embeddings extracted: 161 subjects
Embedding shape per subject: (3, 64)
vae_embeddings.npz saved: 0.16 MB
vae_anomaly_scores.csv saved

ALL OUTPUTS SAVED TO GOOGLE DRIVE:
  best_vae.pth: 17.78 MB
  vae_embeddings.npz: 0.16 MB
  vae_anomaly_scores.csv: 0.01 MB
  vae_loss_curve.png: 0.07 MB
  vae_anomaly_scores.png: 0.06 MB
